# Predicting Life Expectancy – Gapminder (Project 1.2)
Neural Networks and Deep Learning · Winter Semester 2026/27

## 0. Originality & Contributions

**Authors:** (entfernt)

We declare that this notebook was created independently by the authors listed above. All sources used are listed under *References*.

**Contributions:** both authors contributed equally to all parts of the project (data exploration, modelling, training, evaluation and writing).

## 1. Setup

- Runtime: Google Colab (CPU)
- all packages used here are preinstalled in Colab, so nothing has to be installed
- the next cell prints the Python version and the versions of all packages, so the results can be reproduced

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import plotly
import plotly.express as px
import sklearn
import torch
from torch import nn

print("Python", sys.version)
for m in [np, pd, matplotlib, plotly, sklearn, torch]:
    print(f"{m.__name__:12} {m.__version__}")

global settings: random seed, share of the test set, number of epochs and learning rate. the seed is set for numpy and torch so every run gives the same results.

In [ ]:
SEED = 42
TEST_SIZE = 0.2
EPOCHS = 1000
LR = 0.1

np.random.seed(SEED)
torch.manual_seed(SEED);

## 2. Load and Describe Data

the data comes from the **Gapminder Foundation** and is shipped with plotly (`plotly.express.data.gapminder()`). each row is one country in one year.

In [ ]:
df = px.data.gapminder()
df.head()

In [ ]:
df.info()

| column | type | meaning |
|---|---|---|
| country | object | name of the country |
| continent | object | continent (Africa, Americas, Asia, Europe, Oceania) |
| year | int | 1952 to 2007 in 5-year steps |
| lifeExp | float | life expectancy at birth in years → **target** |
| pop | int | population |
| gdpPercap | float | GDP per capita in inflation-adjusted US$ |
| iso_alpha, iso_num | object / int | country codes, only ids |

**target:** `lifeExp` (continuous) → regression.
**features:** `year`, `gdpPercap`, `pop`, `continent`. `country`, `iso_alpha` and `iso_num` only identify the country and are not used as features.

## 3. Exploration

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
print("samples:  ", len(df))
print("countries:", df["country"].nunique())
print("years:    ", df["year"].nunique())

1704 samples (142 countries × 12 years), no missing values. `gdpPercap` and `pop` have a huge range and their mean is much larger than the median, so they are strongly right-skewed.

**distribution of the target:**

In [ ]:
plt.hist(df["lifeExp"], bins=30)
plt.xlabel("lifeExp (years)")
plt.ylabel("count")
plt.title("Distribution of the target")
plt.show()

life expectancy goes from about 24 to 83 years. the distribution is not normal: there is a broad plateau between about 40 and 65 years and a clear peak around 70–72 years (mostly the richer countries in the later years).

**transformation:** because `gdpPercap` and `pop` are very skewed, we take the `log10`. the histograms show the raw and the log-transformed values.

In [ ]:
df["log_gdp"] = np.log10(df["gdpPercap"])
df["log_pop"] = np.log10(df["pop"])

cols = ["gdpPercap", "pop", "log_gdp", "log_pop"]
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, col in zip(axes, cols):
    ax.hist(df[col], bins=30)
    ax.set_title(col)
plt.tight_layout()
plt.show()

after the log transformation both features are much more symmetric, so we use `log_gdp` and `log_pop` from now on.

**correlation matrix:**

In [ ]:
num_cols = ["year", "log_gdp", "log_pop", "lifeExp"]
corr = df[num_cols].corr()

plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(num_cols)), num_cols, rotation=45)
plt.yticks(range(len(num_cols)), num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
plt.title("Correlation matrix")
plt.show()

`log_gdp` has the strongest correlation with `lifeExp` (0.81), `year` is moderate (0.44) and `log_pop` is weak (0.19). the features are hardly correlated with each other. that richer countries have a higher life expectancy is well known as the *Preston curve*.

**feature matrix:** `continent` is a category, so it is one-hot encoded (5 columns). together with `year`, `log_gdp` and `log_pop` we get 8 features.

In [ ]:
X = pd.get_dummies(df[["year", "log_gdp", "log_pop", "continent"]], columns=["continent"], dtype=float)
y = df["lifeExp"].values
print("X:", X.shape, " y:", y.shape)
X.head()

**PCA (unsupervised):** the features are standardized with `StandardScaler` (mean 0, std 1), otherwise `year` (values around 2000) would dominate. then they are projected onto the first two principal components. PCA does not see the target, the color is only added afterwards.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
with np.errstate(all="ignore"):
    pcs = pca.fit_transform(StandardScaler().fit_transform(X))
print("explained variance:", pca.explained_variance_ratio_.round(2))

plt.scatter(pcs[:, 0], pcs[:, 1], c=y, cmap="viridis", s=8)
plt.colorbar(label="lifeExp")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of the features")
plt.show()

the first two components explain only about 44% of the variance, because the 5 one-hot columns add many independent directions. still, the points form clusters (mostly the continents) and the color changes clearly across the plot: one side has low, the other high life expectancy. so the features contain a lot of information about the target.

the scaler here is only for the exploration. for the model it is fitted on the training data only (section 5).

## 4. Model

we model life expectancy as a function of the features, this is a **regression** problem:

$$\text{lifeExp} = f(\text{year}, \log_{10}\text{gdpPercap}, \log_{10}\text{pop}, \text{continent})$$

- **baseline:** linear regression with one layer `nn.Linear(8, 1)`: $\hat{y} = Xw + b$
- **loss:** mean squared error $\text{MSE} = \frac{1}{n}\sum_{i}(y_i - \hat{y}_i)^2$, its root (RMSE) is in years
- **input:** matrix `(n_samples, 8)`, **output:** matrix `(n_samples, 1)`
- **trivial reference:** always predict the mean of the training data

In [ ]:
torch.manual_seed(SEED)
model = nn.Linear(X.shape[1], 1)  # 8 inputs -> 1 output
loss_func = nn.MSELoss()
model

## 5. Training

**split:** every country appears 12 times. with a random row split the model would see e.g. Germany 1952–2002 and only has to guess Germany 2007. so we split the **countries** 80/20: all years of a country are either in train or in test.

In [ ]:
from sklearn.model_selection import train_test_split

train_countries, test_countries = train_test_split(df["country"].unique(), test_size=TEST_SIZE, random_state=SEED)
train_mask = df["country"].isin(train_countries).values

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]
print("train:", X_train.shape, " test:", X_test.shape)

the scaler is fitted **only on the training data**, so no information from the test set leaks into training. then everything is converted to tensors.

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_t = torch.tensor(scaler.transform(X_train), dtype=torch.float32)
X_test_t = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

**training loop:** full-batch gradient descent with plain SGD. we record the train and test loss per epoch, the test loss is only used for monitoring.

In [ ]:
def train(model, X_train, X_test):
    optimizer = torch.optim.SGD(model.parameters(), lr=LR)
    train_hist, test_hist = [], []
    for epoch in range(EPOCHS):
        loss = loss_func(model(X_train), y_train_t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            test_hist.append(loss_func(model(X_test), y_test_t).item())
        train_hist.append(loss.item())

    print(f"final train MSE: {train_hist[-1]:.2f}   final test MSE: {test_hist[-1]:.2f}")
    plt.plot(train_hist, label="train")
    plt.plot(test_hist, label="test")
    plt.yscale("log")
    plt.xlabel("epoch")
    plt.ylabel("MSE loss")
    plt.legend()
    plt.show()

train(model, X_train_t, X_test_t)

the loss drops very fast at the beginning and is flat after about 20 epochs, so the model has converged. the test loss is higher than the train loss because the test countries were never seen.

## 6. Evaluation

- **RMSE:** typical error in years
- **R²:** share of the variance explained (1 = perfect, 0 = as good as the mean)

**why a separate test set?** on the training data the error is too optimistic, a model could just memorise the samples (**overfitting**). only unseen data shows how well the model **generalises** to new countries.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

results = {}

def evaluate(name, y_pred):
    # save the scores and show the table of all models so far
    results[name] = {"RMSE": np.sqrt(mean_squared_error(y_test, y_pred)), "R2": r2_score(y_test, y_pred)}
    return pd.DataFrame(results).T.round(3)

def predict(model, X_test):
    # predict the test set and plot true vs. predicted (red line = perfect prediction)
    with torch.no_grad():
        y_pred = model(X_test).numpy().ravel()
    plt.scatter(y_test, y_pred, s=8, alpha=0.6)
    plt.axline((40, 40), slope=1, color="red", linestyle="--")
    plt.xlabel("true lifeExp")
    plt.ylabel("predicted lifeExp")
    plt.show()
    return y_pred

evaluate("mean (trivial)", np.full(len(y_test), y_train.mean()))

In [ ]:
evaluate("linear", predict(model, X_test_t))

the linear model is much better than the mean: RMSE 13.3 → 6.4 years, R² ≈ 0.77. the largest errors are at low life expectancies (wars, famines, HIV/AIDS), which are not in our features. the errors are also systematic (too high for low values, too low above about 65 years), so the relation is not fully linear.

## 7. Improvement

**polynomial features:** we add all squares and pairwise products of the features (8 → 44), e.g. `log_gdp²` (life expectancy saturates for rich countries) or `log_gdp × continent_Africa` (different GDP effect per continent). the model stays `nn.Linear` with SGD, split, loss and epochs are the same.

the new features are scaled again, otherwise the squares have a much larger spread and SGD diverges.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

poly = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), StandardScaler())
X_train_poly_t = torch.tensor(poly.fit_transform(X_train_t), dtype=torch.float32)
X_test_poly_t = torch.tensor(poly.transform(X_test_t), dtype=torch.float32)

torch.manual_seed(SEED)
model_poly = nn.Linear(X_train_poly_t.shape[1], 1)  # 44 inputs -> 1 output
train(model_poly, X_train_poly_t, X_test_poly_t)
evaluate("polynomial", predict(model_poly, X_test_poly_t))

**does it improve?** only a little: RMSE 6.43 → 6.02 years, R² 0.77 → 0.79. most of the remaining error comes from events that are not in the features (wars, epidemics), so a more flexible model cannot fix it. with only 29 test countries the small difference also depends on which countries are in the test set.

## 8. Summary and Reflection

- **data:** 1704 rows (142 countries × 12 years), no missing values. `gdpPercap` and `pop` are very skewed, so we used `log10`.
- **exploration:** `log_gdp` is the strongest predictor of `lifeExp` (r = 0.81), then `year` (0.44).
- **baseline:** linear regression (`nn.Linear`, SGD, MSE) has a test RMSE of 6.4 years (R² ≈ 0.77), much better than the mean (13.3 years).
- **improvement:** polynomial features help slightly (RMSE 6.0, R² ≈ 0.79).

**assumptions and limits**
- each country appears 12 times, so rows are not independent. we split by country, but the result depends on one random split (cross-validation would be more reliable).
- epochs, learning rate and degree were not tuned on a separate validation set.
- only 4 variables: health care, education, wars or epidemics are missing (e.g. Rwanda 1992).
- **no causality:** we only see correlations between GDP and life expectancy.
- **extrapolation:** data ends in 2007, the model should not be used far beyond that.

## 9. References

- data: https://www.gapminder.org/data/
- dataset in plotly: https://plotly.com/python-api-reference/generated/plotly.express.data.html
- Preston curve: https://en.wikipedia.org/wiki/Preston_curve
- PyTorch: https://docs.pytorch.org/docs/stable/nn.html
- scikit-learn: https://scikit-learn.org/stable/
- numpy: https://numpy.org/doc/stable/
- pandas: https://pandas.pydata.org/docs/
- matplotlib: https://matplotlib.org/stable/
- lecture slides
- gemini 3.5 flash-lite: used for code assistance and debugging